## 18. 🆕 Bootstrap real de $\bar A$ (por vivienda)

La sección 16.3 mostraba un "bootstrap" con el error estándar **escrito a mano** (`se_abar = 0.38`) y un IC calculado como punto ± 1,96·SE: **no había remuestreo**. Esta sección lo hace de verdad con los microdatos de la EPH.

**Qué es un bootstrap.** $\bar A$ sale de una muestra. Si hubiéramos sorteado otra, daría un valor algo distinto; el *error estándar* mide cuánto. Como no podemos volver a sortear la EPH, la **simulamos**: sacamos con reposición, del mismo conjunto de datos, muestras del mismo tamaño y **recalculamos todo** (logit cohorte + edad, extrapolación, $\bar A$) en cada una. La dispersión de los $\bar A$ resultantes es el error estándar; sus percentiles 2,5 y 97,5 dan un IC de 95 %.

**Por vivienda, no por persona.** La EPH rota las viviendas (2 trimestres dentro, 2 fuera, 2 dentro): las *mismas* personas aparecen en varios trimestres. Se remuestrean **viviendas completas** (`CODUSU`) con pesos multinomiales; remuestrear filas sueltas ignoraría esa dependencia y subestimaría la incertidumbre (se muestra en 17.4).

> ⚠️ El IC mide **sólo variabilidad muestral**. No capta error de modelo (perfil por edad común, período/cohorte, EPH urbana, `PP07H` de la semana de referencia, reglas para independientes). Ver 17.7.

### 18.1 Módulo (copia de `src/tbp_bootstrap.py`)

Una sola celda, autosuficiente. Usa las funciones del módulo de la sección 12 (`tbp_eph`): si no se puede importar, toma las que ya están cargadas en el notebook. Terminal: `python src/tbp_bootstrap.py --autotest`.

In [ ]:
# ====== MÓDULO tbp_bootstrap (misma versión que src/tbp_bootstrap.py) ======
# -*- coding: utf-8 -*-
"""
tbp_bootstrap.py — Bootstrap REAL (remuestreo por vivienda) de Ā con los microdatos de la EPH.

Reemplaza el "bootstrap" de la sección 16.3 del notebook principal, que tenía el error estándar escrito a mano
(se = 0,38) y calculaba IC = punto ± 1,96·se sin remuestrear nada.

Idea (para quien no es experta en estadística):
  * Ā sale de una muestra (la EPH). Si hubiéramos sorteado OTRA muestra, Ā daría un valor algo distinto. El "error estándar"
    mide cuánto cambiaría. Como no podemos volver a sortear la EPH, la simulamos: "remuestreamos" con reposición los datos
    que tenemos y recalculamos TODO (logit cohorte+edad, extrapolación, Ā) en cada réplica.
  * Unidad de remuestreo = la VIVIENDA (CODUSU), no la persona ni el trimestre. La EPH rota las viviendas (2 trimestres en la
    muestra, 2 fuera, 2 dentro): las MISMAS personas aparecen en varios trimestres y esas filas no son independientes.
    Remuestrear filas sueltas ignoraría esa dependencia y daría un error estándar demasiado chico (ver `bootstrap_ingenuo`).
  * Para que sea rápido, se precalculan por vivienda las sumas por celda (cohorte, edad): n, Σw, Σw·reg. Cada réplica es una
    suma ponderada con pesos multinomiales de viviendas.

QUÉ CAPTA: sólo la variabilidad MUESTRAL (con el diseño de cluster simplificado). QUÉ NO CAPTA: error de modelo (perfil por
edad común a todas las cohortes, confusión período/cohorte, EPH urbana, PP07H de la semana de referencia, reglas para
independientes, extrapolación "últimas k cohortes", S65 y Gompertz). Ver docs/seccion_18_bootstrap_abar.md.

Uso mínimo:
    import tbp_bootstrap as tb
    df  = tb.read_all_cluster(tb.find_eph_files(["datos/eph"]))        # con CODUSU y AGLOMERADO
    res = tb.bootstrap_abar(df, s65, B=500, seed=20260507)              # s65: Serie cohorte -> S65
    res["tabla"]                                                        # SE, IC95, sesgo por cohorte y regla

NO edita tbp_eph.py: importa de allí el pipeline (add_flags, fit_age_cohort_logit, extrapolar_alpha, densidad_modelada...).
"""
import os, io, sys, time, zipfile, tempfile, types
import numpy as np
import pandas as pd
from scipy import sparse


# ------------------------------------------------------------------ acceso a tbp_eph (módulo o notebook)
_FUNCS = ["add_flags", "fit_age_cohort_logit", "extrapolar_alpha", "densidad_modelada", "supervivencia_gompertz",
          "find_eph_files", "estimar_abar"]
_CONSTS = ["EDADES", "EDAD_MIN", "EDAD_MAX", "MIN_N_CELDA", "MIN_EDADES_COHORTE", "NEEDED", "OBLIGATORIAS"]


def _resolver_eph(ns=None):
    """Devuelve el objeto con las funciones de tbp_eph: `import tbp_eph` o, si no está, las definidas en el notebook."""
    try:
        import tbp_eph
        return tbp_eph
    except ImportError:
        pass
    ns = ns if ns is not None else globals()
    faltan = [n for n in _FUNCS + _CONSTS if n not in ns]
    if faltan:
        raise ImportError("No encuentro tbp_eph (ni importable ni cargado en el notebook). Faltan: %s. "
                          "Corré antes la celda del módulo de la sección 12." % faltan)
    obj = types.SimpleNamespace(**{n: ns[n] for n in _FUNCS + _CONSTS})
    if "ind_rule" not in obj.add_flags.__code__.co_varnames:
        raise ImportError("`add_flags` del notebook no es la de tbp_eph (otra versión). Corré la celda de la sección 12 de nuevo.")
    return obj


_EPH = _resolver_eph()

REGLAS_BS = ("solo_asalariados", "pp07i", "indep50")        # mismos nombres que A_bar_* de estimar_abar
_NQ = 5                                                  # n, w, wreg_solo, wreg_pp07i, wreg_indep


# ================================================================== 1) LECTOR CON CODUSU Y AGLOMERADO
def _leer_texto_cluster(handle):
    cab = handle.readline()
    sep = ";" if cab.count(";") >= cab.count(",") else ","
    handle.seek(0)
    need = set(_EPH.NEEDED) | {"CODUSU", "AGLOMERADO"}
    return pd.read_csv(handle, sep=sep, usecols=lambda c: c.strip().upper() in need, low_memory=False,
                       dtype={"CODUSU": str})


def read_individual_cluster(src):
    """Como tbp_eph.read_individual pero conserva CODUSU (texto, identifica la VIVIENDA) y AGLOMERADO (numérico).
    CODUSU es la vivienda: se repite en los trimestres en que esa vivienda está en la muestra (rotación 2-2-2)."""
    if src["member"] is None:
        with open(src["path"], "r", encoding="latin-1", newline="") as f:
            df = _leer_texto_cluster(f)
    else:
        with zipfile.ZipFile(src["path"]) as z, z.open(src["member"]) as raw:
            df = _leer_texto_cluster(io.StringIO(raw.read().decode("latin-1")))
    df.columns = [c.strip().upper() for c in df.columns]
    for c in df.columns:
        if c != "CODUSU":
            df[c] = pd.to_numeric(df[c], errors="coerce")
    faltan = (set(_EPH.OBLIGATORIAS) | {"CODUSU"}) - set(df.columns)
    if faltan:
        raise ValueError(f"Faltan columnas {sorted(faltan)} en {src}.")
    for opt in ("CAT_OCUP", "PP07H", "PP07I", "CH04", "TRIMESTRE", "AGLOMERADO"):
        if opt not in df.columns:
            df[opt] = np.nan
    df["CODUSU"] = df["CODUSU"].astype(str).str.strip()
    return df


def read_all_cluster(srcs):
    return pd.concat([read_individual_cluster(s) for s in srcs], ignore_index=True)


# ================================================================== 2) PRECÁLCULO POR CLUSTER
def preparar(df, ind_share=0.5, unidad="vivienda", estratificar=False):
    """Precalcula, por unidad de remuestreo, las sumas por celda (cohorte, edad) para las tres reglas.

    unidad = "vivienda" (CODUSU; lo correcto) | "persona" (cada fila es una unidad; bootstrap INGENUO, sólo para comparar).
    estratificar=True: remuestrea viviendas DENTRO de cada aglomerado (el diseño de la EPH es estratificado por aglomerado).
    Devuelve un dict con la matriz dispersa Q (unidades x 5*celdas) y metadatos."""
    d_sol = _EPH.add_flags(df, "solo_asalariados")
    d_pp = _EPH.add_flags(df, "pp07i")
    d_ind = _EPH.add_flags(df, "escenario", ind_share)
    assert d_sol.index.equals(d_pp.index) and d_sol.index.equals(d_ind.index)
    idx = d_sol.index
    if unidad == "vivienda":
        cl, _ = pd.factorize(df.loc[idx, "CODUSU"])
    elif unidad == "persona":
        cl = np.arange(len(idx))
    else:
        raise ValueError(unidad)
    K = int(cl.max()) + 1
    key = d_sol["cohorte"].values.astype(np.int64) * 100 + d_sol["edad"].values.astype(np.int64)
    ukey, cell = np.unique(key, return_inverse=True)
    nc = len(ukey)
    w = d_sol["w"].values
    vals = [np.ones(len(idx)), w, w * d_sol["reg"].values, w * d_pp["reg"].values, w * d_ind["reg"].values]
    rows = np.tile(cl, _NQ)
    cols = np.concatenate([cell + q * nc for q in range(_NQ)])
    Q = sparse.coo_matrix((np.concatenate(vals), (rows, cols)), shape=(K, _NQ * nc)).tocsr()
    est = None
    if estratificar:
        ag = df.loc[idx, "AGLOMERADO"].fillna(-1).values
        est_row = np.zeros(K, dtype=np.int64)
        est_row[cl] = pd.factorize(ag)[0]                  # una vivienda pertenece a un solo aglomerado
        est = est_row
    return {"Q": Q, "K": K, "nc": nc, "cohorte_celda": (ukey // 100).astype(int), "edad_celda": (ukey % 100).astype(int),
            "estratos": est, "unidad": unidad, "n_filas": len(idx), "ind_share": ind_share}


def _pesos(prep, rng):
    """Pesos multinomiales: cuántas veces entra cada unidad en la réplica (suma = K, o = K_h en cada estrato)."""
    K = prep["K"]
    if prep["estratos"] is None:
        return np.bincount(rng.integers(0, K, K), minlength=K).astype(float)
    est = prep["estratos"]; m = np.zeros(K)
    for h in np.unique(est):
        ids = np.flatnonzero(est == h)
        m[ids] = np.bincount(rng.integers(0, len(ids), len(ids)), minlength=len(ids))
    return m


# ================================================================== 3) PIPELINE SOBRE LAS SUMAS (idéntico a tbp_eph.pipeline)
def _abar_desde_sumas(prep, sums, S, cohortes, modo_extrap="ultimas_k", detalle=False):
    """Para cada regla: celdas -> logit cohorte+edad -> extrapolación de alpha -> densidad -> Ā. Misma lógica que
    tbp_eph.build_cells + fit_age_cohort_logit + extrapolar_alpha + densidad_modelada + abar_por_cohorte."""
    nc = prep["nc"]
    n = sums[0:nc]; w = sums[nc:2 * nc]
    out, det = {}, {}
    for q, regla in zip((2, 3, 4), REGLAS_BS):
        wreg = sums[q * nc:(q + 1) * nc]
        keep = (n >= _EPH.MIN_N_CELDA) & (w > 0)
        cells = pd.DataFrame({"cohorte": prep["cohorte_celda"][keep], "edad": prep["edad_celda"][keep], "n": n[keep],
                              "w": w[keep], "wreg": wreg[keep]})
        cells["dens"] = cells["wreg"] / cells["w"]
        fit = _EPH.fit_age_cohort_logit(cells)
        alpha_c = _EPH.extrapolar_alpha(fit, list(cohortes), modo=modo_extrap)
        dens = _EPH.densidad_modelada(fit, alpha_c)
        out[regla] = (S * dens.values).sum(axis=1)
        if detalle:
            det[regla] = {"fit": fit, "dens": dens, "cells": cells}
    return (out, det) if detalle else out


def estimacion_puntual(prep, s65, cohortes=None, modo_extrap="ultimas_k"):
    """Ā con la muestra completa (todas las unidades con peso 1). Debe coincidir con tbp_eph.estimar_abar."""
    cohortes = list(s65.index) if cohortes is None else list(cohortes)
    S = _EPH.supervivencia_gompertz(s65.reindex(cohortes).values)
    sums = np.asarray(prep["Q"].sum(axis=0)).ravel()
    out, det = _abar_desde_sumas(prep, sums, S, cohortes, modo_extrap, detalle=True)
    return pd.DataFrame(out, index=cohortes), det, S


def contribucion_extrapolado(det, S, cohortes):
    """Por cohorte y regla: ¿cuánto de Ā viene de edades OBSERVADAS y cuánto del perfil/modelo (extrapolado)?
    frac_edades_obs = Σ_{a observada} S(a)·d(a) / Ā ; alpha_extrapolado = la cohorte tiene < MIN_EDADES_COHORTE edades
    observadas (su nivel se toma de las últimas k cohortes o de las primeras k)."""
    filas = []
    for regla, dd in det.items():
        cells, fit, dens = dd["cells"], dd["fit"], dd["dens"]
        obs = cells.groupby("cohorte")["edad"].apply(set).to_dict()
        n_ed = fit["n_edades"]
        for i, c in enumerate(cohortes):
            ed = obs.get(c, set())
            mask = np.isin(_EPH.EDADES, list(ed))
            tot = float((S[i] * dens.values[i]).sum())
            parte_obs = float((S[i] * dens.values[i])[mask].sum())
            filas.append({"cohorte": c, "regla": regla, "n_edades_obs": int(len(ed)),
                          "alpha_extrapolado": bool(int(n_ed.get(c, 0)) < _EPH.MIN_EDADES_COHORTE),
                          "frac_edades_obs": parte_obs / tot if tot > 0 else np.nan})
    return pd.DataFrame(filas)


# ================================================================== 4) BOOTSTRAP
_GBS = {}                                                    # estado compartido con los procesos hijos (fork)


def _init_worker(*args):
    if args and args[0] is not None:
        _GBS.update(prep=args[0], S=args[1], coh=args[2], modo=args[3], ss=args[4])


def _una_replica(b):
    prep, S, coh, modo, ss = _GBS["prep"], _GBS["S"], _GBS["coh"], _GBS["modo"], _GBS["ss"]
    rng = np.random.default_rng(ss[b])
    m = _pesos(prep, rng)
    sums = m @ prep["Q"]
    r = _abar_desde_sumas(prep, sums, S, coh, modo)
    return np.concatenate([r[k] for k in REGLAS_BS])


def _correr(prep, s65, cohortes, B, seed, modo_extrap, n_jobs, verbose, etiqueta):
    cohortes = list(cohortes)
    S = _EPH.supervivencia_gompertz(s65.reindex(cohortes).values)
    ss = np.random.SeedSequence(seed).spawn(B)               # una semilla por réplica: el resultado no depende de n_jobs
    _GBS.update(prep=prep, S=S, coh=cohortes, modo=modo_extrap, ss=ss)
    t0 = time.time()
    if n_jobs and n_jobs > 1:
        import multiprocessing as mp
        # En Linux/Colab se usa fork (rápido). En macOS fork + BLAS se cuelga: se usa spawn (requiere tbp_bootstrap importable).
        metodo = "fork" if sys.platform.startswith("linux") else "spawn"
        if metodo == "spawn" and __name__ == "__main__":
            raise RuntimeError("n_jobs>1 en macOS/Windows requiere importar el módulo (import tbp_bootstrap), no correrlo como script.")
        with mp.get_context(metodo).Pool(n_jobs, initializer=_init_worker,
                                         initargs=(prep, S, cohortes, modo_extrap, ss) if metodo == "spawn" else (None,)) as pool:
            filas = pool.map(_una_replica, range(B), chunksize=max(1, B // (n_jobs * 8)))
    else:
        filas = []
        for b in range(B):
            filas.append(_una_replica(b))
            if verbose and (b + 1) % max(1, B // 5) == 0:
                print(f"  [{etiqueta}] réplica {b + 1}/{B}  ({time.time() - t0:.0f} s)")
    arr = np.array(filas).reshape(B, len(REGLAS_BS), len(cohortes))
    return arr, time.time() - t0


def _resumir(punto, arr, cohortes, nivel=0.95):
    a = (1 - nivel) / 2
    filas = []
    for j, regla in enumerate(REGLAS_BS):
        x = arr[:, j, :]
        lo, hi = np.quantile(x, [a, 1 - a], axis=0)
        for i, c in enumerate(cohortes):
            p = float(punto[regla].iloc[i])
            filas.append({"cohorte": c, "regla": regla, "A_punto": p, "se": float(x[:, i].std(ddof=1)),
                          "ic95_lo": float(lo[i]), "ic95_hi": float(hi[i]), "sesgo_boot": float(x[:, i].mean() - p)})
    return pd.DataFrame(filas)


def bootstrap_abar(df, s65, B=1000, seed=20260507, cohortes=None, modo_extrap="ultimas_k", ind_share=0.5,
                   estratificar=False, n_jobs=1, verbose=True, prep=None):
    """Bootstrap por VIVIENDA (CODUSU) de Ā. Reajusta todo el pipeline en cada réplica. Devuelve dict con
    'tabla' (long: cohorte x regla con A_punto, se, ic95_lo, ic95_hi, sesgo_boot, n_edades_obs, alpha_extrapolado,
    frac_edades_obs), 'replicas' (array B x 3 reglas x cohortes), 'punto', 'segundos', 'meta'."""
    t0 = time.time()
    cohortes = list(s65.index) if cohortes is None else list(cohortes)
    prep = prep or preparar(df, ind_share=ind_share, unidad="vivienda", estratificar=estratificar)
    punto, det, S = estimacion_puntual(prep, s65, cohortes, modo_extrap)
    t_prep = time.time() - t0
    arr, seg = _correr(prep, s65, cohortes, B, seed, modo_extrap, n_jobs, verbose, "vivienda")
    tab = _resumir(punto, arr, cohortes).merge(contribucion_extrapolado(det, S, cohortes), on=["cohorte", "regla"])
    meta = {"B": B, "seed": seed, "unidad": prep["unidad"], "estratificado": bool(estratificar), "n_unidades": prep["K"],
            "n_filas": prep["n_filas"], "ind_share": ind_share, "modo_extrap": modo_extrap, "segundos_prep": t_prep,
            "segundos_bootstrap": seg, "n_jobs": n_jobs}
    return {"tabla": tab, "replicas": arr, "punto": punto, "segundos": seg, "meta": meta, "prep": prep}


def bootstrap_ingenuo(df, s65, B=500, seed=20260507, cohortes=None, modo_extrap="ultimas_k", ind_share=0.5,
                      n_jobs=1, verbose=True):
    """Bootstrap INGENUO: remuestrea FILAS (personas-trimestre) como si fueran independientes. Subestima la incertidumbre
    porque ignora que la misma vivienda/persona aparece en varios trimestres. Sólo para mostrar cuánto."""
    cohortes = list(s65.index) if cohortes is None else list(cohortes)
    prep = preparar(df, ind_share=ind_share, unidad="persona")
    punto, det, S = estimacion_puntual(prep, s65, cohortes, modo_extrap)
    arr, seg = _correr(prep, s65, cohortes, B, seed, modo_extrap, n_jobs, verbose, "ingenuo")
    return {"tabla": _resumir(punto, arr, cohortes), "replicas": arr, "punto": punto, "segundos": seg}


# ================================================================== 5) TRASLADO A TBP_C Y TABLA FINAL
def trasladar_a_tbp(tabla, d, tau=0.2177, rho=0.403, tbp_col_p4=None):
    """TBP_C = N·S65·Ā·τ/(J·ρ) es PROPORCIONAL a Ā (N, S65, J, τ, ρ fijos) => los extremos del IC de TBP_C son
    TBP_C(punto) · (límite de Ā / Ā punto). `d`: DataFrame del notebook (cohorte, N_t, S65, J, E65).
    `tbp_col_p4`: opcional, Serie cohorte -> TBP_C de una versión más refinada (p. ej. R15['TBP_C_p4']); si se da, se usa esa base."""
    base = {}
    for regla in REGLAS_BS:
        a = tabla[tabla["regla"] == regla].set_index("cohorte")["A_punto"]
        x = _tbp_abar(d, a, tau, rho)
        base[regla] = x.set_index("cohorte")["TBP_C_Abar"]
    t = tabla.copy()
    t["TBP_C_punto"] = [base[r].get(c, np.nan) for r, c in zip(t["regla"], t["cohorte"])]
    if tbp_col_p4 is not None:
        t["TBP_C_p4_punto"] = [tbp_col_p4.get(c, np.nan) for c in t["cohorte"]]
    for k in ("se", "ic95_lo", "ic95_hi"):
        t[f"TBP_C_{k}"] = t["TBP_C_punto"] * (t[k] / t["A_punto"])
    if tbp_col_p4 is not None:
        for k in ("se", "ic95_lo", "ic95_hi"):
            t[f"TBP_C_p4_{k}"] = t["TBP_C_p4_punto"] * (t[k] / t["A_punto"])
    return t


def _tbp_abar(d, a, tau, rho):
    x = d.set_index("cohorte").join(a.rename("A_bar_eph"), how="left")
    x["TBP_C_Abar"] = x["N_t"] * x["S65"] * x["A_bar_eph"] * tau / (x["J"] * rho)
    return x.reset_index()


def cohortes_base_extrapolacion(det, regla="solo_asalariados", k=5):
    """Cohortes con >= MIN_EDADES_COHORTE edades observadas cuyo efecto alpha se promedia (últimas k) para extrapolar
    a las cohortes jóvenes/futuras (p. ej. la 2024)."""
    fit = det[regla]["fit"]
    ok = fit["alpha"][fit["n_edades"] >= _EPH.MIN_EDADES_COHORTE]
    return list(ok.index[-k:])


def comparar_con_falso(tabla, cohorte=2024, regla="solo_asalariados", se_falso=0.38, punto_falso=14.57):
    """Compara el SE y el IC verdaderos con los escritos a mano (0,38 y [13,82; 15,31])."""
    r = tabla[(tabla["cohorte"] == cohorte) & (tabla["regla"] == regla)].iloc[0]
    falso = (punto_falso - 1.96 * se_falso, punto_falso + 1.96 * se_falso)
    return {"A_punto": r["A_punto"], "se_real": r["se"], "se_falso": se_falso, "razon_se": r["se"] / se_falso,
            "ic_real": (r["ic95_lo"], r["ic95_hi"]), "ic_falso": falso,
            "semiancho_rel_real_%": 100 * (r["ic95_hi"] - r["ic95_lo"]) / 2 / r["A_punto"],
            "semiancho_rel_falso_%": 100 * 1.96 * se_falso / punto_falso}


# ================================================================== 6) DATOS SINTÉTICOS Y AUTOTEST
def _banner_bs(txt="DATOS SINTÉTICOS — no son Argentina"):
    line = "#" * 78
    print("\n".join([line, "##" + txt.center(74) + "##", line]))


def generar_eph_sintetica_cluster(anios=range(2012, 2025), n_viv_por_celda=3, seed=7, efecto_vivienda=0.8):
    """EPH de juguete CON CODUSU: cada vivienda tiene un efecto común (todas sus personas tienden a ser parecidas), así que
    las filas de una misma vivienda están correlacionadas y el bootstrap por vivienda debe dar SE mayor que el ingenuo.
    Las viviendas se repiten en 2 trimestres consecutivos (rotación simplificada). NO son datos de Argentina."""
    rng = np.random.default_rng(seed); filas = []
    nviv = 0
    for y in anios:
        for q in (1, 2, 3, 4):
            for a in range(14, 71):
                for v in range(n_viv_por_celda):
                    nviv += 1
                    u = rng.normal(0, efecto_vivienda)            # efecto de la vivienda
                    for t in (0, 1):                               # la vivienda se entrevista dos trimestres seguidos
                        yy, qq = (y, q + t) if q + t <= 4 else (y + 1, 1)
                        if yy > anios[-1]:
                            continue
                        aa, k = a, 8
                        p = 1 / (1 + np.exp(-(-0.2 + 0.008 * (yy - aa - 1970) + 1.2 * np.exp(-((aa - 40) / 18.0) ** 2) - 1.0 + u)))
                        reg = rng.random(k) < p
                        filas.append(pd.DataFrame({"CODUSU": f"V{nviv:07d}", "AGLOMERADO": (nviv % 5) + 1, "ANO4": yy,
                                                   "TRIMESTRE": qq, "CH04": rng.integers(1, 3, k), "CH06": aa,
                                                   "ESTADO": 1, "CAT_OCUP": 3, "PP07H": np.where(reg, 1, 2), "PP07I": np.nan,
                                                   "PONDERA": rng.integers(300, 900, k)}))
    return pd.concat(filas, ignore_index=True)


def autotest_bootstrap(verbose=True, B=30):
    """Prueba con datos SINTÉTICOS: (1) lector con CODUSU (txt y zip); (2) el punto del bootstrap coincide con
    tbp_eph.estimar_abar; (3) reproducibilidad por semilla y por n_jobs; (4) con efecto de vivienda el SE por vivienda
    es mayor que el ingenuo; (5) con réplicas = muestra completa los pesos multinomiales suman K."""
    if verbose: _banner_bs()
    df = generar_eph_sintetica_cluster()
    coh = np.arange(1990, 2031)
    s65 = pd.Series(np.clip(0.70 + 0.0035 * (coh - 1950), 0.5, 0.97), index=coh)
    # (1) lector
    with tempfile.TemporaryDirectory() as tmp:
        sub = df[df["ANO4"] == 2015].copy()
        sub.columns = [c.lower() if c in ("CH04", "CH06", "ESTADO") else c for c in sub.columns]
        p = os.path.join(tmp, "usu_individual_T115.txt"); sub.to_csv(p, sep=";", index=False)
        zp = os.path.join(tmp, "EPH_usu_1_Trim_2015_txt.zip")
        with zipfile.ZipFile(zp, "w") as z:
            z.write(p, "usu_individual_T115.txt")
        os.remove(p)
        srcs = _EPH.find_eph_files([tmp]); assert len(srcs) == 1
        lec = read_all_cluster(srcs)
        assert len(lec) == len(sub) and {"CODUSU", "AGLOMERADO"} <= set(lec.columns)
        assert lec["CODUSU"].iloc[0].startswith("V") and lec["CODUSU"].nunique() == sub["CODUSU"].nunique()
    # (2) punto == estimar_abar del módulo original
    ref = _EPH.estimar_abar(df.drop(columns=["CODUSU", "AGLOMERADO"]), s65, cohortes=coh)
    prep = preparar(df)
    punto, det, S = estimacion_puntual(prep, s65, coh)
    for regla, col in zip(REGLAS_BS, ("A_bar_solo_asalariados", "A_bar_pp07i", "A_bar_indep50")):
        assert np.allclose(punto[regla].values, ref[col].values, atol=1e-9), f"el punto no coincide con estimar_abar ({regla})"
    # (3) reproducibilidad
    r1 = bootstrap_abar(df, s65, B=B, seed=1, cohortes=coh, verbose=False, prep=prep)
    r2 = bootstrap_abar(df, s65, B=B, seed=1, cohortes=coh, verbose=False, prep=prep)
    r3 = bootstrap_abar(df, s65, B=B, seed=2, cohortes=coh, verbose=False, prep=prep)
    assert np.array_equal(r1["replicas"], r2["replicas"]) and not np.array_equal(r1["replicas"], r3["replicas"])
    if sys.platform.startswith("linux") and "ipykernel" not in sys.modules:   # fork (en macOS se usa spawn: ver _correr)
        r4 = bootstrap_abar(df, s65, B=B, seed=1, cohortes=coh, verbose=False, prep=prep, n_jobs=2)
        assert np.array_equal(r1["replicas"], r4["replicas"]), "el resultado depende de n_jobs"
    # (4) cluster vs ingenuo (el efecto de vivienda y la repetición de viviendas deben ensanchar el SE)
    ing = bootstrap_ingenuo(df, s65, B=B, seed=1, cohortes=coh, verbose=False)
    c = 2005 if 2005 in coh else coh[len(coh) // 2]
    se_cl = r1["tabla"].query("cohorte == @c and regla == 'solo_asalariados'")["se"].iloc[0]
    se_in = ing["tabla"].query("cohorte == @c and regla == 'solo_asalariados'")["se"].iloc[0]
    assert se_cl > se_in, f"con efecto de vivienda el SE por vivienda ({se_cl:.3f}) debería superar al ingenuo ({se_in:.3f})"
    # (5) pesos multinomiales
    m = _pesos(prep, np.random.default_rng(0)); assert m.sum() == prep["K"]
    pe = preparar(df, estratificar=True); m = _pesos(pe, np.random.default_rng(0)); assert m.sum() == pe["K"]
    # IC contiene al punto en la mayoría de cohortes observadas
    t = r1["tabla"]; ok = ((t["ic95_lo"] <= t["A_punto"] + 0.5) & (t["A_punto"] <= t["ic95_hi"] + 0.5)).mean()
    assert ok > 0.9
    if verbose:
        print(f"[sintético] cohorte {c}: SE por vivienda = {se_cl:.3f} | SE ingenuo (por fila) = {se_in:.3f} "
              f"(razón {se_cl / se_in:.2f})")
        print("Autotest OK (lector con CODUSU, punto = estimar_abar, semilla/n_jobs reproducibles, cluster > ingenuo, pesos multinomiales).")
        _banner_bs("FIN AUTOTEST — estos números NO son Argentina")
    return r1

### 18.2 Carga de la EPH con identificador de vivienda

Se leen las mismas columnas de siempre más `CODUSU` (vivienda) y `AGLOMERADO`. Si no hay bases en `datos/eph/` la sección **termina limpia** ("FALTAN DATOS") y no inventa nada; la prueba sintética (17.6) corre igual.

In [ ]:
import time
RUTAS_EPH17 = ["datos/eph", "../datos/eph", "../../datos/eph"]
srcs17 = _EPH.find_eph_files(RUTAS_EPH17)
HAY_EPH17 = len(srcs17) > 0
if not HAY_EPH17:
    print("=" * 78)
    print("FALTAN DATOS: no hay bases individuales de la EPH en", RUTAS_EPH17)
    print("Sin ellas no se calcula ningún bootstrap de Ā. Dejá los .zip de INDEC en datos/eph/ (ver sección 12).")
    print("=" * 78)
else:
    _t0 = time.time()
    df17 = read_all_cluster(srcs17)
    # `d` (tabla de cohortes del notebook principal). Si otra celda lo pisó (p. ej. un `for ..., d in ...`), se relee del CSV que escribió la celda 5.
    _ok = lambda x: isinstance(x, pd.DataFrame) and {"cohorte", "N_t", "S65", "J"} <= set(x.columns)
    d17 = d if _ok(globals().get("d")) else None
    if d17 is None:
        _f = os.path.join(OUT, "tbp_argentina_cohortes_1950_2035.csv")
        if os.path.exists(_f):
            d17 = pd.read_csv(_f); print("Aviso: `d` fue reemplazada por otra celda; se releyó", _f)
    if d17 is None:
        HAY_EPH17 = False
        print("FALTAN DATOS: no encuentro la tabla de cohortes `d` (N_t, S65, J). Corré antes la celda 5 del notebook principal.")
    else:
        s65_17 = d17.set_index("cohorte")["S65"]
    if HAY_EPH17:
        tr = df17.drop_duplicates(["CODUSU", "ANO4", "TRIMESTRE"]).groupby("CODUSU").size()
        print(f"Archivos: {len(srcs17)} | filas: {len(df17):,} | viviendas (CODUSU) distintas: {df17['CODUSU'].nunique():,} | "
              f"aglomerados: {df17['AGLOMERADO'].nunique()} | lectura: {time.time() - _t0:.0f} s")
        print("Trimestres en que aparece cada vivienda (rotación 2-2-2):", tr.value_counts().sort_index().to_dict())
        print(f"=> {100 * (tr > 1).mean():.0f} % de las viviendas aparece en más de un trimestre: las filas NO son independientes.")

### 18.3 Bootstrap por vivienda (B = 1.000) y resultados por cohorte y regla

Cada réplica: pesos multinomiales de viviendas → sumas por celda (cohorte, edad) → logit → extrapolación → $\bar A$, para las tres reglas de independientes. Semilla fija (`20260507`): el resultado es reproducible. La tabla incluye cuánto de $\bar A$ viene de edades **observadas** y cuánto es perfil/extrapolación del modelo.

In [ ]:
if HAY_EPH17:
    B17, SEED17 = 1000, 20260507
    boot17 = bootstrap_abar(df17, s65_17, B=B17, seed=SEED17, verbose=False)
    tab17 = boot17["tabla"]
    print(f"Bootstrap por vivienda: B = {B17} réplicas, {boot17['meta']['n_unidades']:,} viviendas, "
          f"tiempo: {boot17['segundos']:.0f} s (preparación {boot17['meta']['segundos_prep']:.0f} s). Semilla {SEED17}.")
    COH17 = [c for c in (1960, 1980, 2000, 2024) if c in s65_17.index]
    v = tab17[tab17.cohorte.isin(COH17)].copy()
    v["± % (IC95)"] = 100 * (v.ic95_hi - v.ic95_lo) / 2 / v.A_punto
    v["% de Ā en edades observadas"] = 100 * v.frac_edades_obs
    display(v[["cohorte", "regla", "A_punto", "se", "ic95_lo", "ic95_hi", "± % (IC95)", "n_edades_obs", "alpha_extrapolado",
              "% de Ā en edades observadas"]].round(3).reset_index(drop=True))
    _, det17, _ = estimacion_puntual(boot17["prep"], s65_17)
    print("La cohorte 2024 no tiene ninguna edad observada: su nivel (alpha) se extrapola promediando las cohortes",
          cohortes_base_extrapolacion(det17), "(las últimas con >= 3 edades; se las ve sólo a los 18-20 años).")
else:
    print("FALTAN DATOS: sin EPH no hay bootstrap (ver 17.2).")

### 18.4 Comparación honesta: ¿qué había en 16.3?

Se compara el SE verdadero con el `0,38` escrito a mano, el IC verdadero con el `[13,82 ; 15,31]` falso, y con un bootstrap **ingenuo** (por fila) que ignora que las viviendas se repiten. Se agrega una variante estratificada por aglomerado (el diseño de la EPH es estratificado) y un ejemplo de **error de modelo** que el bootstrap no capta (otra regla de extrapolación).

In [ ]:
if HAY_EPH17:
    ing17 = bootstrap_ingenuo(df17, s65_17, B=500, seed=SEED17, verbose=False)
    est17 = bootstrap_abar(df17, s65_17, B=500, seed=SEED17, estratificar=True, verbose=False)
    cmp = comparar_con_falso(tab17, 2024, "solo_asalariados")
    rows = []
    for c in COH17:
        for r in REGLAS_BS:
            f = lambda t, col: t[(t.cohorte == c) & (t.regla == r)][col].iloc[0]
            rows.append({"cohorte": c, "regla": r, "Ā": f(tab17, "A_punto"), "SE ingenuo (filas)": f(ing17["tabla"], "se"),
                         "SE por vivienda": f(tab17, "se"), "SE por vivienda+aglomerado": f(est17["tabla"], "se")})
    cp = pd.DataFrame(rows); cp["vivienda / ingenuo"] = cp["SE por vivienda"] / cp["SE ingenuo (filas)"]
    display(cp.round(3))
    print(f"\nCohorte 2024, regla solo_asalariados (Ā = {cmp['A_punto']:.2f}):")
    print(f"  SE escrito a mano en 16.3: {cmp['se_falso']:.2f}   |   SE real (bootstrap por vivienda): {cmp['se_real']:.2f}   "
          f"(x{cmp['razon_se']:.2f})")
    print(f"  IC 95 % falso: [{cmp['ic_falso'][0]:.2f} ; {cmp['ic_falso'][1]:.2f}] (± {cmp['semiancho_rel_falso_%']:.1f} %)   |   "
          f"IC 95 % real: [{cmp['ic_real'][0]:.2f} ; {cmp['ic_real'][1]:.2f}] (± {cmp['semiancho_rel_real_%']:.1f} %)")
    se_ing24 = cp[(cp.cohorte == 2024) & (cp.regla == "solo_asalariados")]["SE ingenuo (filas)"].iloc[0]
    print(f"  Bootstrap ingenuo (por fila): SE = {se_ing24:.2f} -> subestima el SE real en {100 * (1 - se_ing24 / cmp['se_real']):.0f} %.")
    print("  El supuesto '± 5 %' de 16.3 NO se sostiene: el margen real de 95 % es mayor (ver arriba) y para otras cohortes/reglas varía.")
    alt = estimacion_puntual(boot17["prep"], s65_17, modo_extrap="tendencia")[0]
    a0 = boot17["punto"].loc[2024, "solo_asalariados"]; a1 = alt.loc[2024, "solo_asalariados"]
    dentro = cmp["ic_real"][0] <= a1 <= cmp["ic_real"][1]
    print(f"  Una variante de modelo: Ā 2024 con extrapolación 'tendencia' = {a1:.2f} vs {a0:.2f} con 'últimas 5 cohortes' "
          f"({a1 - a0:+.2f} años, {100 * (a1 / a0 - 1):+.0f} %); queda {'dentro' if dentro else 'fuera'} del IC muestral. "
          "Es UNA variante: el error de modelo (perfil común, período/cohorte...) no se puede cuantificar con estos datos.")
else:
    print("FALTAN DATOS: sin EPH no hay comparación (ver 17.2).")

### 18.5 Traslado a $TBP_C$, gráfico y CSV

$TBP_C=\dfrac{N\,S_{65}\,\bar A\,\tau}{J\,\rho}$ es **proporcional** a $\bar A$ (con $N, S_{65}, J, \tau, \rho$ fijos), así que el IC de $TBP_C$ es el de $\bar A$ multiplicado por $TBP_C/\bar A$: el **margen relativo es el mismo**. Si la sección 15 está cargada (`R15`), se agrega la versión `p4` de 2024 (aproximación: se aplica el margen relativo del $\bar A$ agregado, no el de $\bar A$ por sexo).

In [ ]:
if HAY_EPH17:
    p4 = R15["TBP_C_p4"] if "R15" in globals() and "TBP_C_p4" in getattr(R15, "columns", []) else None
    full17 = trasladar_a_tbp(tab17, d17, tau=globals().get('TAU', 0.2177), rho=globals().get('RHO', 0.403), tbp_col_p4=p4)
    ing_se = ing17["tabla"].rename(columns={"se": "se_ingenuo"})[["cohorte", "regla", "se_ingenuo"]]
    full17 = full17.merge(ing_se, on=["cohorte", "regla"])
    full17["ic95_semiancho_rel_pct"] = 100 * (full17.ic95_hi - full17.ic95_lo) / 2 / full17.A_punto
    full17["B"], full17["semilla"], full17["unidad_remuestreo"] = B17, SEED17, "vivienda (CODUSU)"
    os.makedirs(OUT, exist_ok=True)
    full17.to_csv(f"{OUT}/abar_bootstrap_ic.csv", index=False)
    cols = ["cohorte", "regla", "TBP_C_punto", "TBP_C_se", "TBP_C_ic95_lo", "TBP_C_ic95_hi"]
    display(full17[full17.cohorte.isin(COH17)][cols].round(4).reset_index(drop=True))
    if p4 is not None:
        r24 = full17[(full17.cohorte == 2024) & (full17.regla == "solo_asalariados")].iloc[0]
        print(f"TBP_C p4 2024 (aprox.): {r24.TBP_C_p4_punto:.3f}  IC95 [{r24.TBP_C_p4_ic95_lo:.3f} ; {r24.TBP_C_p4_ic95_hi:.3f}] "
              f"(16.3 mostraba ± 5 %).")
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    col = {"solo_asalariados": "#1f77b4", "pp07i": "#ff7f0e", "indep50": "#2ca02c"}
    for r in REGLAS_BS:
        t = full17[full17.regla == r].sort_values("cohorte")
        ax[0].plot(t.cohorte, t.A_punto, color=col[r], label=r); ax[0].fill_between(t.cohorte, t.ic95_lo, t.ic95_hi, color=col[r], alpha=.2)
    ex = np.sort(full17[(full17.regla == "solo_asalariados") & full17.alpha_extrapolado].cohorte.values)
    if len(ex):                                   # sombrea cada tramo continuo de cohortes con alpha extrapolado
        cortes = np.flatnonzero(np.diff(ex) > 1) + 1
        for k, tramo in enumerate(np.split(ex, cortes)):
            ax[0].axvspan(tramo[0] - .5, tramo[-1] + .5, color="gray", alpha=.15, label="alpha extrapolado (< 3 edades)" if k == 0 else None)
    ax[0].set_title("Ā por cohorte con IC 95 % (bootstrap por vivienda)"); ax[0].set_xlabel("cohorte"); ax[0].set_ylabel("años")
    ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)
    j = REGLAS_BS.index("solo_asalariados"); i = list(s65_17.index).index(2024)
    ax[1].hist(boot17["replicas"][:, j, i], bins=40, color="#4575b4", alpha=.7, edgecolor="white", density=True)
    ax[1].axvline(boot17["punto"].loc[2024, "solo_asalariados"], color="k", label="punto")
    ax[1].axvline(cmp["ic_real"][0], color="#d73027", ls="--", label="IC 95 % real"); ax[1].axvline(cmp["ic_real"][1], color="#d73027", ls="--")
    ax[1].axvline(cmp["ic_falso"][0], color="gray", ls=":", lw=2, label="IC 'falso' de 16.3"); ax[1].axvline(cmp["ic_falso"][1], color="gray", ls=":", lw=2)
    ax[1].set_title("Cohorte 2024, solo asalariados: 1.000 réplicas"); ax[1].set_xlabel("Ā (años)"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.savefig(f"{OUT}/seccion_18_bootstrap_abar.png", dpi=130); plt.show()
    print(f"Guardado: {OUT}/abar_bootstrap_ic.csv ({len(full17)} filas) y {OUT}/seccion_18_bootstrap_abar.png")
else:
    print("FALTAN DATOS: sin EPH no hay CSV de bootstrap (ver 17.2).")

### 18.6 Prueba con datos sintéticos (autotest)

Verifica con una EPH **inventada** (con viviendas que se repiten y un efecto de vivienda): lector con `CODUSU` (`.txt` y `.zip`), que el punto coincide con `estimar_abar`, que la semilla reproduce el resultado, que los pesos multinomiales suman bien y que el SE por vivienda supera al ingenuo. **No son datos de Argentina.**

In [ ]:
_ = autotest_bootstrap()

### 18.7 Lectura y límites

* **Qué se puede afirmar.** El error estándar muestral de $\bar A$ (cohorte 2024, solo asalariados) es **mayor** que el `0,38` escrito a mano en 16.3 (ver 17.4), y el margen de 95 % **no es ± 5 %**. El `0,38` no salía de ningún cálculo. Ignorar que las viviendas se repiten (bootstrap por fila) subestima el SE entre un cuarto y un tercio (según cohorte y regla).
* **Qué no se puede afirmar.** Que este IC sea "la" incertidumbre de $\bar A$ ni de $TBP_C$. Mide **sólo variabilidad muestral** de la EPH con un diseño simplificado (se remuestrean viviendas; los ponderadores se toman como fijos; no se usa el estrato ni la UPM del INDEC).
* **No capta error de modelo:** perfil por edad común a todas las cohortes; confusión período/cohorte; extrapolación de las cohortes jóvenes y futuras (la 2024 no tiene edades observadas; ver el ejemplo de 17.4); EPH urbana; `PP07H` de la semana de referencia; reglas para independientes (por eso se reportan tres); $S_{65}$ y Gompertz tratados como exactos.
* **Para cohortes viejas casi todo $\bar A$ es perfil de edad, no dato:** la columna "% de Ā en edades observadas" es baja (7–24 % en las cohortes mostradas), así que $\bar A$ depende sobre todo del perfil común $f(a)$, cuya incertidumbre muestral sí está en el SE pero cuya forma funcional no.
* **$TBP_C$:** el IC se traslada por proporcionalidad; no incluye incertidumbre de $N$, $\tau$, $\rho$, $J$ (eso es el Monte Carlo de 16.4, que además usa un rango de $\bar A$ propio).